In [1]:
import sys
import os
sys.path.append(os.path.abspath("../.."))

from app.data_providers import test_train_dataset
from processing.compute_columns import add_shot_main_action_type_column

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import TargetEncoder
from sklearn.preprocessing import FunctionTransformer

from sklearn.calibration import CalibrationDisplay

from sklearn.pipeline import FeatureUnion
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_transformer

from sklearn.metrics import roc_auc_score, f1_score
from sklearn.metrics import classification_report

from imblearn.over_sampling import RandomOverSampler

from xgboost import XGBClassifier

import joblib

from copy import deepcopy

import shap

pd.set_option('display.max_columns', None)

In [2]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    log_loss,
    brier_score_loss
)

import tensorflow as tf
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    BatchNormalization,
    Embedding,
    Flatten,
    Concatenate
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# Data preparation

In [3]:
data_train = pd.read_parquet(r"../../data/trajectory_tests/train_new_all_players_v3.parquet")
data_train.set_index('Unnamed: 0', inplace=True)
data_train = data_train.rename_axis(index=None, axis=1)

data_test = pd.read_parquet(r"../../data/trajectory_tests/test_new_all_players_v3.parquet")
data_test.set_index('Unnamed: 0', inplace=True)
data_test = data_test.rename_axis(index=None, axis=1)

In [4]:
data_train.head()

,shooter_slot,shooter_x,shooter_y,shooter_team_id,shot_angle,distance_to_basket_tracking,nearest_defender_dist,avg_defender_dist,defenders_within_3ft,defenders_within_5ft,defenders_within_7ft,offensive_spacing_area,shooter_speed,player1_speed,player2_speed,player3_speed,player4_speed,player5_speed,player6_speed,player7_speed,player8_speed,player9_speed,player10_speed,defender_closing_speed,ball_height,ball_speed,ball_xy_speed,ACTION_TYPE,EVENTTIME,EVENT_TYPE,GAME_DATE,GAME_EVENT_ID,GAME_ID,GRID_TYPE,HTM,LOC_X,LOC_Y,MINUTES_REMAINING,PERIOD,PLAYER_ID,PLAYER_NAME,QUARTER,SECONDS_REMAINING,SHOT_ATTEMPTED_FLAG,SHOT_DISTANCE,SHOT_MADE_FLAG,SHOT_TIME,SHOT_TYPE,SHOT_ZONE_AREA,SHOT_ZONE_BASIC,SHOT_ZONE_RANGE,TEAM_ID,TEAM_NAME,VTM,tracking_game_clock,tracking_game_clock_old,event_GAME_ID,event_EVENTNUM,event_EVENTMSGTYPE,event_EVENTMSGACTIONTYPE,event_PERIOD,event_WCTIMESTRING,event_PCTIMESTRING,event_HOMEDESCRIPTION,event_NEUTRALDESCRIPTION,event_VISITORDESCRIPTION,event_SCORE,event_SCOREMARGIN,event_PERSON1TYPE,event_PLAYER1_ID,event_PLAYER1_NAME,event_PLAYER1_TEAM_ID,event_PLAYER1_TEAM_CITY,event_PLAYER1_TEAM_NICKNAME,event_PLAYER1_TEAM_ABBREVIATION,event_PERSON2TYPE,event_PLAYER2_ID,event_PLAYER2_NAME,event_PLAYER2_TEAM_ID,event_PLAYER2_TEAM_CITY,event_PLAYER2_TEAM_NICKNAME,event_PLAYER2_TEAM_ABBREVIATION,event_PERSON3TYPE,event_PLAYER3_ID,event_PLAYER3_NAME,event_PLAYER3_TEAM_ID,event_PLAYER3_TEAM_CITY,event_PLAYER3_TEAM_NICKNAME,event_PLAYER3_TEAM_ABBREVIATION,ball_x,ball_y,ball_z,player1_team_id,player1_id,player1_x,player1_y,player2_team_id,player2_id,player2_x,player2_y,player3_team_id,player3_id,player3_x,player3_y,player4_team_id,player4_id,player4_x,player4_y,player5_team_id,player5_id,player5_x,player5_y,player6_team_id,player6_id,player6_x,player6_y,player7_team_id,player7_id,player7_x,player7_y,player8_team_id,player8_id,player8_x,player8_y,player9_team_id,player9_id,player9_x,player9_y,player10_team_id,player10_id,player10_x,player10_y,shooter_x_t0,shooter_y_t0,defender1_dx_t0,defender1_dy_t0,defender1_dist_t0,defender2_dx_t0,defender2_dy_t0,defender2_dist_t0,defender3_dx_t0,defender3_dy_t0,defender3_dist_t0,defender4_dx_t0,defender4_dy_t0,defender4_dist_t0,defender5_dx_t0,defender5_dy_t0,defender5_dist_t0,attacker1_dx_t0,attacker1_dy_t0,attacker1_dist_t0,attacker2_dx_t0,attacker2_dy_t0,attacker2_dist_t0,attacker3_dx_t0,attacker3_dy_t0,attacker3_dist_t0,attacker4_dx_t0,attacker4_dy_t0,attacker4_dist_t0,shooter_x_t1,shooter_y_t1,defender1_dx_t1,defender1_dy_t1,defender1_dist_t1,defender2_dx_t1,defender2_dy_t1,defender2_dist_t1,defender3_dx_t1,defender3_dy_t1,defender3_dist_t1,defender4_dx_t1,defender4_dy_t1,defender4_dist_t1,defender5_dx_t1,defender5_dy_t1,defender5_dist_t1,attacker1_dx_t1,attacker1_dy_t1,attacker1_dist_t1,attacker2_dx_t1,attacker2_dy_t1,attacker2_dist_t1,attacker3_dx_t1,attacker3_dy_t1,attacker3_dist_t1,attacker4_dx_t1,attacker4_dy_t1,attacker4_dist_t1,shooter_x_t2,shooter_y_t2,defender1_dx_t2,defender1_dy_t2,defender1_dist_t2,defender2_dx_t2,defender2_dy_t2,defender2_dist_t2,defender3_dx_t2,defender3_dy_t2,defender3_dist_t2,defender4_dx_t2,defender4_dy_t2,defender4_dist_t2,defender5_dx_t2,defender5_dy_t2,defender5_dist_t2,attacker1_dx_t2,attacker1_dy_t2,attacker1_dist_t2,attacker2_dx_t2,attacker2_dy_t2,attacker2_dist_t2,attacker3_dx_t2,attacker3_dy_t2,attacker3_dist_t2,attacker4_dx_t2,attacker4_dy_t2,attacker4_dist_t2,shooter_x_t3,shooter_y_t3,defender1_dx_t3,defender1_dy_t3,defender1_dist_t3,defender2_dx_t3,defender2_dy_t3,defender2_dist_t3,defender3_dx_t3,defender3_dy_t3,defender3_dist_t3,defender4_dx_t3,defender4_dy_t3,defender4_dist_t3,defender5_dx_t3,defender5_dy_t3,defender5_dist_t3,attacker1_dx_t3,attacker1_dy_t3,attacker1_dist_t3,attacker2_dx_t3,attacker2_dy_t3,attacker2_dist_t3,attacker3_dx_t3,attacker3_dy_t3,attacker3_dist_t3,attacker4_dx_t3,attacker4_dy_t3,attacker4_dist_t3,shooter_x_t4,shooter_y_t4,defender1_dx_t4,defender1_dy_t4,defender1_dist_t4,defender2_dx_t4,defender2_dy_t4,defender2_d

In [5]:
# Add Main Action type
data_train = add_shot_main_action_type_column(data_train)
data_test = add_shot_main_action_type_column(data_test)

In [6]:
data_train.shape

(44403, 314)

In [7]:
data_test.shape

(13107, 314)

# Define columns

In [8]:
# TARGET
TARGET = "SHOT_MADE_FLAG"  


# FEATURE GROUPS
tracking_features = [
    # t0
    "shooter_x_t0", "shooter_y_t0",
    "defender1_dx_t0", "defender1_dy_t0", "defender1_dist_t0",
    "defender2_dx_t0", "defender2_dy_t0", "defender2_dist_t0",
    "defender3_dx_t0", "defender3_dy_t0", "defender3_dist_t0",
    "defender4_dx_t0", "defender4_dy_t0", "defender4_dist_t0",
    "defender5_dx_t0", "defender5_dy_t0", "defender5_dist_t0",
    "attacker1_dx_t0", "attacker1_dy_t0", "attacker1_dist_t0",
    "attacker2_dx_t0", "attacker2_dy_t0", "attacker2_dist_t0",
    "attacker3_dx_t0", "attacker3_dy_t0", "attacker3_dist_t0",
    "attacker4_dx_t0", "attacker4_dy_t0", "attacker4_dist_t0",

    # t1
    "shooter_x_t1", "shooter_y_t1",
    "defender1_dx_t1", "defender1_dy_t1", "defender1_dist_t1",
    "defender2_dx_t1", "defender2_dy_t1", "defender2_dist_t1",
    "defender3_dx_t1", "defender3_dy_t1", "defender3_dist_t1",
    "defender4_dx_t1", "defender4_dy_t1", "defender4_dist_t1",
    "defender5_dx_t1", "defender5_dy_t1", "defender5_dist_t1",
    "attacker1_dx_t1", "attacker1_dy_t1", "attacker1_dist_t1",
    "attacker2_dx_t1", "attacker2_dy_t1", "attacker2_dist_t1",
    "attacker3_dx_t1", "attacker3_dy_t1", "attacker3_dist_t1",
    "attacker4_dx_t1", "attacker4_dy_t1", "attacker4_dist_t1",

    # t2
    "shooter_x_t2", "shooter_y_t2",
    "defender1_dx_t2", "defender1_dy_t2", "defender1_dist_t2",
    "defender2_dx_t2", "defender2_dy_t2", "defender2_dist_t2",
    "defender3_dx_t2", "defender3_dy_t2", "defender3_dist_t2",
    "defender4_dx_t2", "defender4_dy_t2", "defender4_dist_t2",
    "defender5_dx_t2", "defender5_dy_t2", "defender5_dist_t2",
    "attacker1_dx_t2", "attacker1_dy_t2", "attacker1_dist_t2",
    "attacker2_dx_t2", "attacker2_dy_t2", "attacker2_dist_t2",
    "attacker3_dx_t2", "attacker3_dy_t2", "attacker3_dist_t2",
    "attacker4_dx_t2", "attacker4_dy_t2", "attacker4_dist_t2",

    # t3
    "shooter_x_t3", "shooter_y_t3",
    "defender1_dx_t3", "defender1_dy_t3", "defender1_dist_t3",
    "defender2_dx_t3", "defender2_dy_t3", "defender2_dist_t3",
    "defender3_dx_t3", "defender3_dy_t3", "defender3_dist_t3",
    "defender4_dx_t3", "defender4_dy_t3", "defender4_dist_t3",
    "defender5_dx_t3", "defender5_dy_t3", "defender5_dist_t3",
    "attacker1_dx_t3", "attacker1_dy_t3", "attacker1_dist_t3",
    "attacker2_dx_t3", "attacker2_dy_t3", "attacker2_dist_t3",
    "attacker3_dx_t3", "attacker3_dy_t3", "attacker3_dist_t3",
    "attacker4_dx_t3", "attacker4_dy_t3", "attacker4_dist_t3",

    # t4
    "shooter_x_t4", "shooter_y_t4",
    "defender1_dx_t4", "defender1_dy_t4", "defender1_dist_t4",
    "defender2_dx_t4", "defender2_dy_t4", "defender2_dist_t4",
    "defender3_dx_t4", "defender3_dy_t4", "defender3_dist_t4",
    "defender4_dx_t4", "defender4_dy_t4", "defender4_dist_t4",
    "defender5_dx_t4", "defender5_dy_t4", "defender5_dist_t4",
    "attacker1_dx_t4", "attacker1_dy_t4", "attacker1_dist_t4",
    "attacker2_dx_t4", "attacker2_dy_t4", "attacker2_dist_t4",
    "attacker3_dx_t4", "attacker3_dy_t4", "attacker3_dist_t4",
    "attacker4_dx_t4", "attacker4_dy_t4", "attacker4_dist_t4",

    # t5
    "shooter_x_t5", "shooter_y_t5",
    "defender1_dx_t5", "defender1_dy_t5", "defender1_dist_t5",
    "defender2_dx_t5", "defender2_dy_t5", "defender2_dist_t5",
    "defender3_dx_t5", "defender3_dy_t5", "defender3_dist_t5",
    "defender4_dx_t5", "defender4_dy_t5", "defender4_dist_t5",
    "defender5_dx_t5", "defender5_dy_t5", "defender5_dist_t5",
    "attacker1_dx_t5", "attacker1_dy_t5", "attacker1_dist_t5",
    "attacker2_dx_t5", "attacker2_dy_t5", "attacker2_dist_t5",
    "attacker3_dx_t5", "attacker3_dy_t5", "attacker3_dist_t5",
    "attacker4_dx_t5", "attacker4_dy_t5", "attacker4_dist_t5",
]

additional_features = [
    "shot_angle",
    "distance_to_basket_tracking",
    "nearest_defender_dist",
    "avg_defender_dist",
    "defenders_within_3ft",
    "defenders_within_5ft",
    "defenders_within_7ft",
    "defender_closing_speed",
    "defenders_between",
    "has_screen",
    "teammate_between_defender",
    "players_in_paint",
    "shooter_speed",
    "nearest_teammate_distance"
]

continuous_features = (
    tracking_features +
    additional_features
)

categorical_features = [
    "PERIOD",
    "MAIN_ACTION_TYPE"
]

player_feature = "PLAYER_ID"


In [9]:
required_columns = (
    continuous_features +
    categorical_features +
    [player_feature, TARGET]
)

data_train = data_train.dropna(subset=required_columns).copy()
data_test = data_test.dropna(subset=required_columns).copy()

# Label encoding for categorical and ID features

In [10]:
# Label encode MAIN_ACTION_TYPE
UNKNOWN_ACTION_IDX = 0
unique_actions = data_train['MAIN_ACTION_TYPE'].unique()

action_to_idx = {
    action: idx + 1
    for idx, action in enumerate(unique_actions)
}

data_train['MAIN_ACTION_TYPE'] = (
    data_train['MAIN_ACTION_TYPE']
    .map(action_to_idx)
    .fillna(UNKNOWN_ACTION_IDX)
    .astype(int)
)

data_test['MAIN_ACTION_TYPE'] = (
    data_test['MAIN_ACTION_TYPE']
    .map(action_to_idx)
    .fillna(UNKNOWN_ACTION_IDX)
    .astype(int)
)

num_actions = max(action_to_idx.values()) + 1

In [11]:
# LABEL ENCODE PLAYER IDS

unique_players = data_train[player_feature].unique()

player_to_idx = {
    player_id: idx + 1
    for idx, player_id in enumerate(unique_players)
}

UNKNOWN_PLAYER_IDX = 0


# Encode Train/Test
data_train[player_feature] = (
    data_train[player_feature]
    .map(player_to_idx)
    .fillna(UNKNOWN_PLAYER_IDX)
    .astype(int)
)

data_test[player_feature] = (
    data_test[player_feature]
    .map(player_to_idx)
    .fillna(UNKNOWN_PLAYER_IDX)
    .astype(int)
)

num_players = max(player_to_idx.values()) + 1







In [12]:
# TRAIN / VALIDATION / TEST SPLIT
# We need validation for early stopping

X_train = data_train.drop('SHOT_MADE_FLAG', axis=1)
y_train = data_train['SHOT_MADE_FLAG']

X_test = data_test.drop('SHOT_MADE_FLAG', axis=1)
y_test = data_test['SHOT_MADE_FLAG']

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

# Feature definition

In [13]:
# CONTINUOUS FEATURES
X_train_cont = X_train[continuous_features].values
X_valid_cont = X_val[continuous_features].values
X_test_cont = X_test[continuous_features].values


# STANDARDIZATION
scaler = StandardScaler()

X_train_cont = scaler.fit_transform(X_train_cont)
X_valid_cont = scaler.transform(X_valid_cont)
X_test_cont = scaler.transform(X_test_cont)


# PERIOD INPUT
X_train_period = X_train["PERIOD"].values
X_valid_period = X_val["PERIOD"].values
X_test_period = X_test["PERIOD"].values

# MAIN ACTION TYPE Input
X_train_action = X_train["MAIN_ACTION_TYPE"].values
X_valid_action = X_val["MAIN_ACTION_TYPE"].values
X_test_action = X_test["MAIN_ACTION_TYPE"].values

# PLAYER INPUT
X_train_player = X_train[player_feature].values
X_valid_player = X_val[player_feature].values
X_test_player = X_test[player_feature].values


# TARGET
y_train = y_train.values
y_valid = y_val.values
y_test = y_test.values

# Model Definition

In [14]:
# ==========================
# MODEL
# ==========================

# Continuous Input
continuous_input = Input(
    shape=(len(continuous_features),),
    name="continuous_input"
)

x = Dense(512, activation="relu")(continuous_input)
x = BatchNormalization()(x)
x = Dropout(0.20)(x)

x = Dense(256, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.20)(x)

x = Dense(128, activation="relu")(x)
x = BatchNormalization()(x)


# PLAYER EMBEDDING
player_input = Input(
    shape=(1,),
    name="player_input"
)

player_embedding = Embedding(
    input_dim=num_players + 1,
    output_dim=16,
    name="player_embedding"
)(player_input)

player_embedding = Flatten()(player_embedding)

# PERIOD EMBEDDING
period_input = Input(
    shape=(1,),
    name="period_input"
)

period_embedding = Embedding(
    input_dim=10,
    output_dim=4,
    name="period_embedding"
)(period_input)

period_embedding = Flatten()(period_embedding)

# ACTION TYPE EMBEDDING
action_input = Input(
    shape=(1,),
    name="action_input"
)

action_embedding = Embedding(
    input_dim=num_actions,
    output_dim=4,
    name="action_embedding"
)(action_input)

action_embedding = Flatten()(action_embedding)


# CONCATENATE
x = Concatenate()([
    x,
    player_embedding,
    period_embedding,
    action_embedding
])

x = Dense(64, activation="relu")(x)
x = Dropout(0.20)(x)

output = Dense(
    1,
    activation="sigmoid"
)(x)


# ========================
# BUILD MODEL
# ========================

model = Model(
    inputs=[
        continuous_input,
        player_input,
        period_input,
        action_input
    ],
    outputs=output
)

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.AUC(name="auc")
    ]
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ continuous_input (InputLayer) │ (None, 188)               │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 512)               │          96,768 │ continuous_input[0][0]     │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization           │ (None, 512)               │           2,048 │ dense[0][0]                │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout (Dropout)             │ (None, 512)               │               0 │ batch_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_1 (Dense)               │ (None, 256)               │         131,328 │ dropout[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_1         │ (None, 256)               │           1,024 │ dense_1[0][0]              │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_1 (Dropout)           │ (None, 256)               │               0 │ batch_normalization_1[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ player_input (InputLayer)     │ (None, 1)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ period_input (InputLayer)     │ (None, 1)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ action_input (InputLayer)     │ (None, 1)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_2 (Dense)               │ (None, 128)               │          32,896 │ dropout_1[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ player_embedding (Embedding)  │ (None, 1, 16)             │           6,640 │ player_input[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ period_embedding (Embedding)  │ (None, 1, 4)              │              40 │ period_input[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ action_embedding (Embedding)  │ (None, 1, 4)              │              24 │ action_input[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_2         │ (None, 128)               │             512 │ dense_2[0][0]              │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ flatten (Flatten)             │ (None, 16)                │               

 Total params: 281,137 (1.07 MB)

 Trainable params: 279,345 (1.07 MB)

 Non-trainable params: 1,792 (7.00 KB)

In [15]:
# Early stopping callback

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

In [16]:
# TRAIN

history = model.fit(
    x=[
        X_train_cont,
        X_train_player,
        X_train_period,
        X_train_action
    ],
    y=y_train,

    validation_data=(
        [
            X_valid_cont,
            X_valid_player,
            X_valid_period,
            X_valid_action
        ],
        y_valid
    ),

    epochs=100,
    batch_size=256,
    callbacks=[early_stopping],
    verbose=1
)




Epoch 1/100
155/155 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - auc: 0.5599 - loss: 0.7133 - val_auc: 0.6093 - val_loss: 0.6682
Epoch 2/100
155/155 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - auc: 0.6278 - loss: 0.6590 - val_auc: 0.6171 - val_loss: 0.6564
Epoch 3/100
155/155 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - auc: 0.6420 - loss: 0.6492 - val_auc: 0.6264 - val_loss: 0.6535
Epoch 4/100
155/155 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - auc: 0.6527 - loss: 0.6435 - val_auc: 0.6309 - val_loss: 0.6534
Epoch 5/100
155/155 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - auc: 0.6603 - loss: 0.6395 - val_auc: 0.6273 - val_loss: 0.6542
Epoch 6/100
155/155 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - auc: 0.6643 - loss: 0.6374 - val_auc: 0.6258 - val_loss: 0.6533
Epoch 7/100
155/155 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - auc: 0.6719 - loss: 0.6333 - val_auc: 0.6291 - val_loss: 0.6525
Epoch 8/100
155/155 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - auc: 0.6714 - loss: 0.6335 - val_auc: 0.6307 - val_loss: 0.6531
Epoch 9/100
155/155 ━━━━━━━━━━━━━━━━━━━━

In [17]:
# PREDICTIONS

y_pred_proba = model.predict([
    X_test_cont,
    X_test_player,
    X_test_period,
    X_test_action
]).flatten()

y_pred = (y_pred_proba >= 0.5).astype(int)


# EVALUATION

auc = roc_auc_score(y_test, y_pred_proba)
ll = log_loss(y_test, y_pred_proba)
brier = brier_score_loss(y_test, y_pred_proba)

print(f"AUC:        {auc:.4f}")
print(f"Log Loss:   {ll:.4f}")
print(f"Brier:      {brier:.4f}")

405/405 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
AUC:        0.6406
Log Loss:   0.6466
Brier:      0.2284


In [18]:
display(pd.crosstab(y_test, y_pred))
print(classification_report(y_test, y_pred))

col_0,0,1
row_0,,
0,5737,1417
1,3463,2332


              precision    recall  f1-score   support

           0       0.62      0.80      0.70      7154
           1       0.62      0.40      0.49      5795

    accuracy                           0.62     12949
   macro avg       0.62      0.60      0.60     12949
weighted avg       0.62      0.62      0.61     12949



In [19]:
# Build error Dataframe

error_df = X_test.copy()

error_df["y_true"] = y_test
error_df["y_pred"] = y_pred
error_df["y_pred_proba"] = y_pred_proba

# confidence
error_df["confidence"] = np.where(
    error_df["y_pred"] == 1,
    error_df["y_pred_proba"],
    1 - error_df["y_pred_proba"]
)

error_df["correct"] = (
    error_df["y_true"] == error_df["y_pred"]
)

# prediction error magnitude
error_df["error"] = abs(error_df["y_true"] - error_df["y_pred_proba"])

In [20]:
for action_type in error_df["MAIN_ACTION_TYPE"].value_counts().index[:15]:

    subset = error_df[
        error_df["MAIN_ACTION_TYPE"] == action_type
    ]

    accuracy = (
        subset["y_true"] == subset["y_pred"]
    ).mean()

    print("\n" + "="*60)
    print(f"{action_type}")
    print(f"n = {len(subset)}")
    print(f"accuracy = {accuracy:.3f}")
    print(f"roc-auc = {roc_auc_score(subset["y_true"], subset["y_pred_proba"]):.3f}")
    print("="*60)

    cm = pd.crosstab(
        subset["y_true"],
        subset["y_pred"],
        rownames=["Actual"],
        colnames=["Predicted"]
    )

    display(cm)


2
n = 8303
accuracy = 0.629
roc-auc = 0.543


Predicted,0,1
Actual,,
0,5061,240
1,2839,163



1
n = 3299
accuracy = 0.580
roc-auc = 0.585


Predicted,0,1
Actual,,
0,492,975
1,409,1423



5
n = 646
accuracy = 0.904
roc-auc = 0.642


Predicted,1
Actual,
0,62
1,584



4
n = 511
accuracy = 0.481
roc-auc = 0.514


Predicted,0,1
Actual,,
0,153,85
1,180,93



3
n = 190
accuracy = 0.526
roc-auc = 0.528


Predicted,0,1
Actual,,
0,31,55
1,35,69


In [22]:
# save model for visualization and inference
model.save("deepNN.keras")
joblib.dump(scaler, "deepNN_scaler.pkl")
joblib.dump(continuous_features, "deepNN_continuous_features.pkl")
joblib.dump(player_to_idx, "deepNN_player_to_idx.pkl")
joblib.dump(action_to_idx, "deepNN_action_to_idx.pkl")
data_test.to_parquet("example_shots.parquet")

# XGBoost for reference

In [25]:
data_train.head()

,shooter_slot,shooter_x,shooter_y,shooter_team_id,shot_angle,distance_to_basket_tracking,nearest_defender_dist,avg_defender_dist,defenders_within_3ft,defenders_within_5ft,defenders_within_7ft,offensive_spacing_area,shooter_speed,player1_speed,player2_speed,player3_speed,player4_speed,player5_speed,player6_speed,player7_speed,player8_speed,player9_speed,player10_speed,defender_closing_speed,ball_height,ball_speed,ball_xy_speed,ACTION_TYPE,EVENTTIME,EVENT_TYPE,GAME_DATE,GAME_EVENT_ID,GAME_ID,GRID_TYPE,HTM,LOC_X,LOC_Y,MINUTES_REMAINING,PERIOD,PLAYER_ID,PLAYER_NAME,QUARTER,SECONDS_REMAINING,SHOT_ATTEMPTED_FLAG,SHOT_DISTANCE,SHOT_MADE_FLAG,SHOT_TIME,SHOT_TYPE,SHOT_ZONE_AREA,SHOT_ZONE_BASIC,SHOT_ZONE_RANGE,TEAM_ID,TEAM_NAME,VTM,tracking_game_clock,tracking_game_clock_old,event_GAME_ID,event_EVENTNUM,event_EVENTMSGTYPE,event_EVENTMSGACTIONTYPE,event_PERIOD,event_WCTIMESTRING,event_PCTIMESTRING,event_HOMEDESCRIPTION,event_NEUTRALDESCRIPTION,event_VISITORDESCRIPTION,event_SCORE,event_SCOREMARGIN,event_PERSON1TYPE,event_PLAYER1_ID,event_PLAYER1_NAME,event_PLAYER1_TEAM_ID,event_PLAYER1_TEAM_CITY,event_PLAYER1_TEAM_NICKNAME,event_PLAYER1_TEAM_ABBREVIATION,event_PERSON2TYPE,event_PLAYER2_ID,event_PLAYER2_NAME,event_PLAYER2_TEAM_ID,event_PLAYER2_TEAM_CITY,event_PLAYER2_TEAM_NICKNAME,event_PLAYER2_TEAM_ABBREVIATION,event_PERSON3TYPE,event_PLAYER3_ID,event_PLAYER3_NAME,event_PLAYER3_TEAM_ID,event_PLAYER3_TEAM_CITY,event_PLAYER3_TEAM_NICKNAME,event_PLAYER3_TEAM_ABBREVIATION,ball_x,ball_y,ball_z,player1_team_id,player1_id,player1_x,player1_y,player2_team_id,player2_id,player2_x,player2_y,player3_team_id,player3_id,player3_x,player3_y,player4_team_id,player4_id,player4_x,player4_y,player5_team_id,player5_id,player5_x,player5_y,player6_team_id,player6_id,player6_x,player6_y,player7_team_id,player7_id,player7_x,player7_y,player8_team_id,player8_id,player8_x,player8_y,player9_team_id,player9_id,player9_x,player9_y,player10_team_id,player10_id,player10_x,player10_y,shooter_x_t0,shooter_y_t0,defender1_dx_t0,defender1_dy_t0,defender1_dist_t0,defender2_dx_t0,defender2_dy_t0,defender2_dist_t0,defender3_dx_t0,defender3_dy_t0,defender3_dist_t0,defender4_dx_t0,defender4_dy_t0,defender4_dist_t0,defender5_dx_t0,defender5_dy_t0,defender5_dist_t0,attacker1_dx_t0,attacker1_dy_t0,attacker1_dist_t0,attacker2_dx_t0,attacker2_dy_t0,attacker2_dist_t0,attacker3_dx_t0,attacker3_dy_t0,attacker3_dist_t0,attacker4_dx_t0,attacker4_dy_t0,attacker4_dist_t0,shooter_x_t1,shooter_y_t1,defender1_dx_t1,defender1_dy_t1,defender1_dist_t1,defender2_dx_t1,defender2_dy_t1,defender2_dist_t1,defender3_dx_t1,defender3_dy_t1,defender3_dist_t1,defender4_dx_t1,defender4_dy_t1,defender4_dist_t1,defender5_dx_t1,defender5_dy_t1,defender5_dist_t1,attacker1_dx_t1,attacker1_dy_t1,attacker1_dist_t1,attacker2_dx_t1,attacker2_dy_t1,attacker2_dist_t1,attacker3_dx_t1,attacker3_dy_t1,attacker3_dist_t1,attacker4_dx_t1,attacker4_dy_t1,attacker4_dist_t1,shooter_x_t2,shooter_y_t2,defender1_dx_t2,defender1_dy_t2,defender1_dist_t2,defender2_dx_t2,defender2_dy_t2,defender2_dist_t2,defender3_dx_t2,defender3_dy_t2,defender3_dist_t2,defender4_dx_t2,defender4_dy_t2,defender4_dist_t2,defender5_dx_t2,defender5_dy_t2,defender5_dist_t2,attacker1_dx_t2,attacker1_dy_t2,attacker1_dist_t2,attacker2_dx_t2,attacker2_dy_t2,attacker2_dist_t2,attacker3_dx_t2,attacker3_dy_t2,attacker3_dist_t2,attacker4_dx_t2,attacker4_dy_t2,attacker4_dist_t2,shooter_x_t3,shooter_y_t3,defender1_dx_t3,defender1_dy_t3,defender1_dist_t3,defender2_dx_t3,defender2_dy_t3,defender2_dist_t3,defender3_dx_t3,defender3_dy_t3,defender3_dist_t3,defender4_dx_t3,defender4_dy_t3,defender4_dist_t3,defender5_dx_t3,defender5_dy_t3,defender5_dist_t3,attacker1_dx_t3,attacker1_dy_t3,attacker1_dist_t3,attacker2_dx_t3,attacker2_dy_t3,attacker2_dist_t3,attacker3_dx_t3,attacker3_dy_t3,attacker3_dist_t3,attacker4_dx_t3,attacker4_dy_t3,attacker4_dist_t3,shooter_x_t4,shooter_y_t4,defender1_dx_t4,defender1_dy_t4,defender1_dist_t4,defender2_dx_t4,defender2_dy_t4,defender2_d

In [26]:
feature_set_reduced = ['MAIN_ACTION_TYPE']
feature_set_reduced.extend(['shooter_x', 'shooter_y', 'shot_angle', 'distance_to_basket_tracking', 'nearest_defender_dist',
    'avg_defender_dist', 'defenders_within_3ft', 'defenders_within_5ft', 'defenders_within_7ft',
    'shooter_speed', 'defender_closing_speed',
    'nearest_teammate_distance', 'defenders_between', 'has_screen', 'offensive_spacing', 'teammate_between_defender', 'players_in_paint'])

cols_OneHot = ['MAIN_ACTION_TYPE']

In [27]:
encoder_oh = OneHotEncoder(sparse_output=False, drop='first', handle_unknown="ignore")
column_encoder = make_column_transformer(
    (encoder_oh, cols_OneHot),
    remainder='passthrough',
    verbose_feature_names_out=False
)
column_encoder.set_output(transform="pandas")

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('onehotencoder', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g

In [28]:
def select_features(df_):
    return df_[feature_set_reduced]

pipeline = Pipeline([
    ("feature_selection",
     FunctionTransformer(select_features)),
    ("encoding",
     column_encoder),
    ("model",
     XGBClassifier(
         n_estimators=500,
         max_depth=3,
         learning_rate=0.01,
         subsample=0.7,
         min_child_weight=1,
         gamma=0,
         random_state=42,
         n_jobs=-1,
         eval_metric="logloss"
     ))
])
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('feature_selection', ...), ('encoding', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function sel...002E78D2CEE80>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` method is only defined if`feature_names_out` is not None.See ``get_feature_names_out`` for more details... versionadded:: 1.1",None
,"kw_args kw_args: dict, default=NoneDictionary of additional keyword argum

In [29]:
# General metrics

y_pred_proba = pipeline.predict_proba(X_test)
y_pred = pipeline.predict(X_test)

print(roc_auc_score(y_test, y_pred_proba[:,1]))
display(pd.crosstab(y_test, y_pred))
print(classification_report(y_test, y_pred))

print("Brier score: ", brier_score_loss(y_test, y_pred_proba[:,1]))

0.6414395079482738


col_0,0,1
row_0,,
0,5557,1597
1,3249,2546


              precision    recall  f1-score   support

           0       0.63      0.78      0.70      7154
           1       0.61      0.44      0.51      5795

    accuracy                           0.63     12949
   macro avg       0.62      0.61      0.60     12949
weighted avg       0.62      0.63      0.61     12949

Brier score:  0.2275005578994751


# Lookup table for reference

In [43]:
train_lookup = X_train.copy()

train_lookup["y"] = y_train.values

lookup_table = (
    train_lookup
    .groupby(
        ["MAIN_ACTION_TYPE", "PLAYER_NAME"]
    )["y"]
    .mean()
    .reset_index()
    .rename(columns={"y": "p_make"})
)

lookup_table.head()

lookup_dict = {
    (
        row["MAIN_ACTION_TYPE"],
        row["PLAYER_NAME"]
    ): row["p_make"]
    
    for _, row in lookup_table.iterrows()
}

In [44]:
global_mean = y_train.mean()

def lookup_predict(X):

    probs = []

    for _, row in X.iterrows():

        key = (
            row["MAIN_ACTION_TYPE"],
            row["PLAYER_NAME"]
        )

        # fallback to global mean if unseen
        p = lookup_dict.get(key, global_mean)

        probs.append(p)

    probs = np.array(probs)

    preds = (probs >= 0.5).astype(int)

    return preds, probs

In [45]:
y_pred_lookup, y_proba_lookup = lookup_predict(X_test)

from sklearn.metrics import (
    roc_auc_score,
    classification_report
)

print(
    "ROC-AUC:",
    roc_auc_score(y_test, y_proba_lookup)
)

print(
    classification_report(
        y_test,
        y_pred_lookup
    )
)

print("Brier score: ", brier_score_loss(y_test, y_proba_lookup))

ROC-AUC: 0.6209452563750335
              precision    recall  f1-score   support

           0       0.61      0.80      0.69      7154
           1       0.61      0.38      0.47      5795

    accuracy                           0.61     12949
   macro avg       0.61      0.59      0.58     12949
weighted avg       0.61      0.61      0.59     12949

Brier score:  0.2354969990430738
